In [353]:
import pandapipes as pp
from tespy.components import Compressor,SimpleHeatExchanger,CycleCloser, Valve, HeatExchanger,Condenser, Sink, Source
from tespy.connections import Connection
from tespy.networks import Network
import numpy as np
from tespy.tools import UserDefinedEquation


class Bidirectional_W_to_WHeatPump:

    def __init__( self,name,net,refrigerant,HC_heat_consumer_id):
        self.name = name
        self.refrigerant = refrigerant
        self.HC_heat_consumer_id=HC_heat_consumer_id
        self.net=net
        self._build_tespy_Cycles()

    def _build_tespy_Cycles(self):
        def my_ude(ude):
            dt_DHN=ude.params['dt']
            return ude.conns[0].calc_T()+dt_DHN-ude.conns[1].calc_T()
        def my_ude_dependents(ude):
            c1, c2 = ude.conns
            return [c1.p,c1.h, c2.p,c2.h]
        self.nw_Cooling_net = Network()
        self.nw_Cooling_net.units.set_defaults(temperature="degC", pressure="bar",pressure_difference="bar",  enthalpy="J/kg", heat="W", power="W")
        self.Cooling_net_compressor = Compressor("compresor")
        self.Cooling_net_condenser = Condenser("condensador")
        self.Cooling_net_valve = Valve("valvula_expansion")
        self.Cooling_net_evaporator = HeatExchanger("evaporador")
        self.Cooling_net_cc=CycleCloser('CycleCloser')
        self.Cooling_net_source_consumer=Source("Source_Consumer ")
        self.Cooling_net_sink_consumer=Sink("Sink_consumer")
        self.Cooling_net_source_reseau=Source("Source_Reseau")
        self.Cooling_net_sink_reseau=Sink("Sink_Reseau")
        self.Cooling_net_c0=Connection(self.Cooling_net_valve, 'out1', self.Cooling_net_cc, 'in1', label='0')
        self.Cooling_net_c1 = Connection(self.Cooling_net_cc, 'out1', self.Cooling_net_evaporator, 'in2', label='1')
        self.Cooling_net_c2 = Connection(self.Cooling_net_evaporator, 'out2', self.Cooling_net_compressor, 'in1', label='2')
        self.Cooling_net_c3 = Connection(self.Cooling_net_compressor, 'out1', self.Cooling_net_condenser, 'in1', label='3')
        self.Cooling_net_c4 = Connection(self.Cooling_net_condenser, 'out1', self.Cooling_net_valve, 'in1', label='4')
        self.Cooling_net_c5=Connection(self.Cooling_net_condenser, 'out2',self.Cooling_net_sink_consumer, 'in1', label='5')
        self.Cooling_net_c6=Connection(self.Cooling_net_source_consumer, 'out1',self.Cooling_net_condenser , 'in2', label='6')
        self.Cooling_net_c7=Connection(self.Cooling_net_source_reseau, 'out1',self.Cooling_net_evaporator , 'in1', label='7')
        self.Cooling_net_c8=Connection(self.Cooling_net_evaporator, 'out1',self.Cooling_net_sink_reseau , 'in1', label='8')
        self.nw_Cooling_net.add_conns(self.Cooling_net_c0,  self.Cooling_net_c1,  self.Cooling_net_c2,  self.Cooling_net_c3,  self.Cooling_net_c4 , self.Cooling_net_c5,  self.Cooling_net_c6, self.Cooling_net_c7, self.Cooling_net_c8)
        self.Cooling_ude = UserDefinedEquation(
                    'my_ude', my_ude, my_ude_dependents, conns=[self.Cooling_net_c8, self.Cooling_net_c7],params={'dt': 5})
        self.nw_Cooling_net.add_ude(self.Cooling_ude)
        
        #Heating net 
        self.nw_Heating_net = Network()
        self.nw_Heating_net.units.set_defaults(temperature="degC", pressure="bar",pressure_difference="bar",  enthalpy="J/kg", heat="W", power="W")
        self.Heating_net_compressor = Compressor("compresor")
        self.Heating_net_condenser = Condenser("condensador")
        self.Heating_net_valve = Valve("valvula_expansion")
        self.Heating_net_evaporator = HeatExchanger("evaporador")
        self.Heating_net_cc=CycleCloser('CycleCloser')
        self.Heating_net_source_consumer=Source("Source_Consumer ")
        self.Heating_net_sink_consumer=Sink("Sink_consumer")
        self.Heating_net_source_reseau=Source("Source_Reseau")
        self.Heating_net_sink_reseau=Sink("Sink_Reseau")
        self.Heating_net_c0=Connection(self.Heating_net_valve, 'out1', self.Heating_net_cc, 'in1', label='0')
        self.Heating_net_c1 = Connection(self.Heating_net_cc, 'out1', self.Heating_net_evaporator, 'in2', label='1')
        self.Heating_net_c2 = Connection(self.Heating_net_evaporator, 'out2', self.Heating_net_compressor, 'in1', label='2')
        self.Heating_net_c3 = Connection(self.Heating_net_compressor, 'out1', self.Heating_net_condenser, 'in1', label='3')
        self.Heating_net_c4 = Connection(self.Heating_net_condenser, 'out1', self.Heating_net_valve, 'in1', label='4')
        self.Heating_net_c5 = Connection(self.Heating_net_evaporator, 'out1', self.Heating_net_sink_consumer, 'in1', label='5')
        self.Heating_net_c6 = Connection(self.Heating_net_source_consumer, 'out1', self.Heating_net_evaporator, 'in1', label='6')  
        self.Heating_net_c7 = Connection(self.Heating_net_source_reseau, 'out1', self.Heating_net_condenser, 'in2', label='7')
        self.Heating_net_c8 = Connection(self.Heating_net_condenser, 'out2', self.Heating_net_sink_reseau, 'in1', label='8')
        self.nw_Heating_net.add_conns(self.Heating_net_c0,  self.Heating_net_c1,  self.Heating_net_c2,  self.Heating_net_c3,  self.Heating_net_c4 , self.Heating_net_c5,  self.Heating_net_c6, self.Heating_net_c7, self.Heating_net_c8)
        self.Heating_ude = UserDefinedEquation(
                        'my_ude_4', my_ude, my_ude_dependents, conns=[self.Heating_net_c7, self.Heating_net_c8],params={'dt': 5})
        self.nw_Heating_net.add_ude(self.Heating_ude)
        
    def solve_cycle(self,mode,Q_consumer,eta_s,T_nework_in,T_cons,dt_DHN):
        #The pressure values for the consumer and district heating side in the heating pump are arbitrary values
        self.mode=mode
        T_cons_in = T_cons[0]
        T_cons_out = T_cons[1]
        T_DH_in = T_nework_in
        if self.mode=="COOLING_NET":
            self.Cooling_net_evaporator.set_attr(pr1=1, pr2=1, ttd_l=5)
            self.Cooling_net_condenser.set_attr(pr1=1, pr2=1, ttd_u=5,Q=Q_consumer)
            self.Cooling_net_compressor.set_attr(eta_s=eta_s)
            self.Cooling_net_c2.set_attr(fluid={self.refrigerant: 1},td_dew=5)
            # 6. Parámetros del Consumidor 
            self.Cooling_net_c5.set_attr(T=T_cons_out, p=3, fluid={"water": 1})
            self.Cooling_net_c6.set_attr(T=T_cons_in)
            # 7. Parámetros de la Red de Distrito (Recibe calor, se calienta de 50 °C a 60 °C)
            self.Cooling_net_c7.set_attr(T=T_DH_in, p=2.5, fluid={"water": 1})
            import CoolProp.CoolProp as CP
            T_triple = CP.Props1SI("Ttriple", self.refrigerant)        # Triple point temperature (K)
            p_triple = CP.Props1SI("ptriple", self.refrigerant)        # Triple point pressure (Pa)
            T_critical = CP.Props1SI("T_critical", self.refrigerant)  # Critical temperature (K)
            p_critical = CP.Props1SI("p_critical", self.refrigerant)  # Critical pressure (Pa)
            h_min = CP.PropsSI("H", "T", T_triple + 0.1, "Q", 0, self.refrigerant)
            T_max_K =  T_critical*0.9
            p_high = min(p_critical * 0.9, 30e5) 
            h_max = CP.PropsSI("H", "T", T_max_K, "P", p_high, self.refrigerant)
            self.nw_Cooling_net._set_p_range([p_triple, p_high])
            self.nw_Cooling_net._set_h_range([h_min,h_max])
            self.Cooling_ude.params['dt'] = dt_DHN
            self.nw_Cooling_net.solve('design')        
        elif self.mode=="HEATING_NET":
            self.Heating_net_evaporator.set_attr(pr1=1, pr2=1, ttd_l=5,Q=Q_consumer)
            self.Heating_net_condenser.set_attr(pr1=1, pr2=1, ttd_u=5)
            self.Heating_net_compressor.set_attr(eta_s=eta_s)
            self.Heating_net_c2.set_attr(fluid={self.refrigerant: 1},td_dew=5)
            # 6. Parámetros del Consumidor 
            self.Heating_net_c5.set_attr(T=T_cons_out, p=3, fluid={"water": 1})
            self.Heating_net_c6.set_attr(T=T_cons_in)
            # 7. Parámetros de la Red de Distrito (Recibe calor, se calienta de 50 °C a 60 °C)
            self.Heating_net_c7.set_attr(T=T_DH_in, p=2.5, fluid={"water": 1})
            import CoolProp.CoolProp as CP
            T_triple = CP.Props1SI("Ttriple", self.refrigerant)        # Triple point temperature (K)
            p_triple = CP.Props1SI("ptriple", self.refrigerant)        # Triple point pressure (Pa)
            T_critical = CP.Props1SI("T_critical", self.refrigerant)  # Critical temperature (K)
            p_critical = CP.Props1SI("p_critical", self.refrigerant)  # Critical pressure (Pa)
            h_min = CP.PropsSI("H", "T", T_triple + 0.1, "Q", 0, self.refrigerant)
            T_max_K =  T_critical*0.9
            p_high = min(p_critical * 0.9, 30e5) 
            h_max = CP.PropsSI("H", "T", T_max_K, "P", p_high, self.refrigerant)
            self.nw_Heating_net._set_p_range([p_triple, p_high])
            self.nw_Heating_net._set_h_range([h_min,h_max])
            self.Heating_ude.params['dt'] = dt_DHN
            self.nw_Heating_net.solve('design')
    def interaction_simulation(self,dT_water):
        if self.mode=="COOLING_NET":
            self.net.heat_consumer.at[self.HC_heat_consumer_id, "qext_w"] =abs( self.Cooling_net_evaporator.Q.val)
            self.net.heat_consumer.at[self.HC_heat_consumer_id,"deltat_k"]=dT_water
            
        elif self.mode=="HEATING_NET":
            self.net.heat_consumer.at[self.HC_heat_consumer_id, "qext_w"] = self.Heating_net_condenser.Q.val
            self.net.heat_consumer.at[self.HC_heat_consumer_id,"deltat_k"]=dT_water

        else: 
            self.net.heat_consumer.at[self.HC_heat_consumer_id, "in_service"] =False
            self.net.heat_consumer.at[self.HC_heat_consumer_id, "in_service"] =False

    def get_main_results(self):
        if self.mode=="COOLING_NET":
            results = {
                "Q_consumer": abs(self.Cooling_net_condenser.Q.val),
                "Q_network": abs(self.Cooling_net_evaporator.Q.val),
                "W_compressor":self.Cooling_net_compressor.P.val ,
                "COP": abs(self.Cooling_net_condenser.Q.val)/self.Cooling_net_compressor.P.val 
            }
        elif self.mode=="HEATING_NET":
            results = {
                "Q_consumer": abs(self.Heating_net_evaporator.Q.val),
                "Q_network": abs(self.Heating_net_condenser.Q.val),
                "COP": abs(self.Heating_net_evaporator.Q.val)/self.Heating_net_compressor.P.val 
            }
        else:
            results = {}
        return results

    def get_results_solver(self):
        if self.mode=="COOLING_NET":
            self.nw_Cooling_net.print_results()
        elif self.mode=="HEATING_NET":
            self.nw_Heating_net.print_results()
        else:
            return None


In [354]:
net = pp.create_empty_network(fluid="water")
# Nudos de la Central / Fuente
j_fuente_ida = pp.create_junction(net, pn_bar=5.0, tfluid_k=353.15, name="Fuente_Ida")
j_nodo_1 = pp.create_junction(net, pn_bar=5.0, tfluid_k=353.15, name="nodo_1_ida")
j_nodo_2=pp.create_junction(net, pn_bar=5.0, tfluid_k=353.15, name="nodo_2_ida") 
j_HP_1 = pp.create_junction(net, pn_bar=5.0, tfluid_k=353.15, name="HP_1_Ida")
j_HP_2 = pp.create_junction(net, pn_bar=5.0, tfluid_k=353.15, name="HP_2_Ida")
#Return
j_fuente_ret = pp.create_junction(net, pn_bar=1.5, tfluid_k=333.15, name="Fuente_Retorno")
j_nodo_1_ret = pp.create_junction(net, pn_bar=5.0, tfluid_k=353.15, name="Node_1_ret")
j_nodo_2_ret=pp.create_junction(net, pn_bar=5.0, tfluid_k=353.15, name="Nodo_2_ret") 
j_HP_ret_1= pp.create_junction(net, pn_bar=5.0, tfluid_k=353.15, name="HP_1_ret")
j_HP_ret_2=pp.create_junction(net, pn_bar=5.0, tfluid_k=353.15, name="HP_2_ret")

#Plant-Nodo
u=0.35 / (np.pi * 0.15)
t_ext=273.15+25
#Planta 1
pipe_ida_1 = pp.create_pipe_from_parameters(
    net, from_junction=j_fuente_ida, to_junction=j_nodo_1,
    length_km=6.8, inner_diameter_mm=1000, u_w_per_m2k=u, text_k=t_ext, name="Tubo_Ida_Plant_Nodo_1",k_mm=0.1
)

pipe_retorno_1= pp.create_pipe_from_parameters(
    net, from_junction=j_nodo_1_ret, to_junction=j_fuente_ret,
    length_km=6.8, inner_diameter_mm=1000, u_w_per_m2k=u,  text_k=t_ext, name="Tubo_RetornoPlant_Nodo_1",k_mm=0.1)

#Nodo-nodo
pipe_ida_2 = pp.create_pipe_from_parameters(
    net, from_junction=j_nodo_1, to_junction=j_nodo_2,
    length_km=4.7, inner_diameter_mm=1000, u_w_per_m2k=u,  text_k=t_ext, name="nodo_1_nodo_2",k_mm=0.1)
pipe_retorno_2= pp.create_pipe_from_parameters(
    net, from_junction=j_nodo_2_ret, to_junction=j_nodo_1_ret,
    length_km=4.7, inner_diameter_mm=1000, u_w_per_m2k=u,  text_k=t_ext, name="nodo_2_ret_nodo_1_ret",k_mm=0.1
)

#Node HP
#Node_Cons1

pipe_loop_3= pp.create_pipe_from_parameters(
    net, from_junction=j_nodo_2, to_junction=j_HP_1,
    length_km=4.6, inner_diameter_mm=1000, u_w_per_m2k=u,  text_k=t_ext, name="Tubo_Ida_nodo_2_cons_1_ida",k_mm=0.1
)
pipe_loop_4= pp.create_pipe_from_parameters(
    net, from_junction=j_HP_1, to_junction=j_HP_ret_1,
    length_km=0.002, inner_diameter_mm=1000, u_w_per_m2k=u,  text_k=t_ext, name="Tubo_bypass_1",k_mm=0.1
)

pipe_loop_5 = pp.create_pipe_from_parameters(
    net, from_junction=j_HP_ret_1, to_junction=j_HP_2,
    length_km=4.7, inner_diameter_mm=1000, u_w_per_m2k=u,  text_k=t_ext, name="Tubo_Ida_cons_1_cons_2_ida",k_mm=0.1
)
pipe_loop_6 = pp.create_pipe_from_parameters(
    net, from_junction=j_HP_2, to_junction=j_HP_ret_2,
    length_km=0.002, inner_diameter_mm=1000, u_w_per_m2k=u,  text_k=t_ext, name="Tubo_bypass_2",k_mm=0.1
)

pipe_loop_7= pp.create_pipe_from_parameters(
    net, from_junction=j_HP_ret_2, to_junction=j_nodo_2_ret,
    length_km=4.5, inner_diameter_mm=1000, u_w_per_m2k=u,  text_k=t_ext, name="Tubo_Retorno_nodo_2_cons_2_ret",k_mm=0.1
)

#Heat Pumps
#Heat pump: extraction
HP_1=pp.create_heat_consumer(
    net,
    from_junction=j_HP_1 ,
    to_junction=j_HP_ret_1,
    qext_w=150000,
    deltat_k=50,
    name="HP_1_EXT"
)
#Heat pump: extraction
HP_2=pp.create_heat_consumer(
    net,
    from_junction=j_HP_2 ,
    to_junction=j_HP_ret_2,
    qext_w=-150000,
    deltat_k=-50,
    name="HP_2_EXT"
)
#Plant: 
Plant_1=pp.create_circ_pump_const_mass_flow(net,flow_junction=j_fuente_ida,return_junction=j_fuente_ret,p_flow_bar=3.5,mdot_flow_kg_per_s=750,t_flow_k=310 ,name='Grid')

In [355]:
T_network_inital_guess_1=45
T_network_inital_guess_2=39


In [356]:
HP_1_object=Bidirectional_W_to_WHeatPump(name="heat_pump_1",refrigerant="R134a",net=net,HC_heat_consumer_id=HP_1)
HP_2_object=Bidirectional_W_to_WHeatPump(name="heat_pump_2",refrigerant="R134a",net=net,HC_heat_consumer_id=HP_2)

In [357]:
T_network_HP_1=T_network_inital_guess_1
T_network_HP_2=T_network_inital_guess_2
T_network_HP_1_loop=T_network_inital_guess_1
T_network_HP_2_loop=T_network_inital_guess_2
tolerance=1E-6
error_1=10
error_2=10
iterations=0
while error_1>tolerance or error_2>tolerance:
    HP_1_object.solve_cycle(mode="COOLING_NET",Q_consumer=-15000,eta_s=0.95,T_nework_in=T_network_HP_1,T_cons=[50,60],dt_DHN=5)
    HP_2_object.solve_cycle(mode="HEATING_NET",Q_consumer=-15000000,eta_s=0.95,T_nework_in=T_network_HP_2,T_cons=[35,25],dt_DHN=10)
    print("iteration")
    HP_1_object.interaction_simulation(dT_water=5)
    HP_2_object.interaction_simulation(dT_water=-10)
    pp.pipeflow(net=net,mode='bidirectional')
    T_network_HP_1=float(net.res_heat_consumer.at[HP_1_object.HC_heat_consumer_id,"t_from_k"]-273.15)
    T_network_HP_2=float(net.res_heat_consumer.at[HP_2_object.HC_heat_consumer_id,"t_from_k"]-273.15)
    error_1=abs(T_network_HP_1_loop-T_network_HP_1)
    error_2=abs(T_network_HP_2_loop-T_network_HP_2)
    print(T_network_HP_1)
    print(T_network_HP_2)
    T_network_HP_1_loop=T_network_HP_1
    T_network_HP_2_loop=T_network_HP_2
    iterations+=1


 iter  | residual   | progress   | massflow   | pressure   | enthalpy   | fluid      | energy     | component  
-------+------------+------------+------------+------------+------------+------------+------------+------------
 1     | 8.37e+05   | 0 %        | 1.71e+00   | 1.02e+06   | 3.60e+05   | 0.00e+00   | 0.00e+00   | 0.00e+00   
 2     | 2.82e+04   | 17 %       | 9.60e-01   | 4.07e+04   | 4.62e+03   | 0.00e+00   | 0.00e+00   | 0.00e+00   
 3     | 3.90e+03   | 26 %       | 1.89e-01   | 7.37e+02   | 1.50e+01   | 0.00e+00   | 0.00e+00   | 0.00e+00   
 4     | 1.46e-02   | 87 %       | 6.53e-07   | 2.30e-01   | 4.61e-03   | 0.00e+00   | 0.00e+00   | 0.00e+00   
 5     | 5.72e-05   | 100 %      | 8.14e-13   | 2.24e-08   | 5.40e-05   | 0.00e+00   | 0.00e+00   | 0.00e+00   
 6     | 1.03e-09   | 100 %      | 7.47e-11   | 4.94e-09   | 2.46e-06   | 0.00e+00   | 0.00e+00   | 0.00e+00   
Total iterations: 6, Calculation time: 0.03 s, Iterations per second: 212.16

 iter  | residual   | pro

In [358]:
net.heat_consumer

,name,from_junction,to_junction,qext_w,controlled_mdot_kg_per_s,deltat_k,treturn_k,in_service,type
0,HP_1_EXT,3,8,1.272033e+04,NaN,5.0,NaN,True,heat_consumer
1,HP_2_EXT,4,9,-1.709219e+07,NaN,-10.0,NaN,True,heat_consumer


In [359]:
net.res_heat_consumer

,p_from_bar,p_to_bar,t_from_k,t_to_k,t_outlet_k,mdot_from_kg_per_s,mdot_to_kg_per_s,vdot_m3_per_s,deltat_k,qext_w
0,2.611709,2.611599,309.858841,309.854766,304.858841,0.608645,-0.608645,0.000612,5.0,1.272033e+04
1,2.352292,2.352269,309.813890,315.265659,319.813890,408.872015,-408.872015,0.412420,-10.0,-1.709219e+07


In [360]:
net.heat_consumer

,name,from_junction,to_junction,qext_w,controlled_mdot_kg_per_s,deltat_k,treturn_k,in_service,type
0,HP_1_EXT,3,8,1.272033e+04,NaN,5.0,NaN,True,heat_consumer
1,HP_2_EXT,4,9,-1.709219e+07,NaN,-10.0,NaN,True,heat_consumer


In [361]:
net.res_heat_consumer

,p_from_bar,p_to_bar,t_from_k,t_to_k,t_outlet_k,mdot_from_kg_per_s,mdot_to_kg_per_s,vdot_m3_per_s,deltat_k,qext_w
0,2.611709,2.611599,309.858841,309.854766,304.858841,0.608645,-0.608645,0.000612,5.0,1.272033e+04
1,2.352292,2.352269,309.813890,315.265659,319.813890,408.872015,-408.872015,0.412420,-10.0,-1.709219e+07


In [362]:
net.circ_pump_pressure

,name,return_junction,flow_junction,p_flow_bar,t_flow_k,plift_bar,in_service,type


In [363]:
net.res_circ_pump_mass

,p_from_bar,p_to_bar,t_from_k,t_to_k,t_outlet_k,mdot_from_kg_per_s,mdot_to_kg_per_s,vdot_m3_per_s,deltat_k,qext_w
0,1.468055,3.5,315.063045,310.0,310.0,750.0,-750.0,0.755783,5.063045,-1.587216e+07


In [364]:
net.ext_grid

,name,junction,p_bar,t_k,in_service,type


In [365]:
net.res_ext_grid

,mdot_kg_per_s


In [366]:
net.res_junction

,p_bar,t_k
0,3.500000,310.000000
1,3.124817,309.940173
2,2.865503,309.898999
3,2.611709,309.858841
4,2.352292,309.813890
5,1.468055,315.063045
6,1.843838,315.148862
7,2.103577,315.208430
8,2.611599,309.854766
9,2.352269,315.265659


In [367]:
net.pipe

,name,from_junction,to_junction,std_type,length_km,inner_diameter_mm,outer_diameter_mm,k_mm,loss_coefficient,u_w_per_m2k,text_k,sections,in_service,type
0,Tubo_Ida_Plant_Nodo_1,0,1,None,6.800,1000.0,1000.0,0.1,0.0,0.742723,298.15,1,True,pipe
1,Tubo_RetornoPlant_Nodo_1,6,5,None,6.800,1000.0,1000.0,0.1,0.0,0.742723,298.15,1,True,pipe
2,nodo_1_nodo_2,1,2,None,4.700,1000.0,1000.0,0.1,0.0,0.742723,298.15,1,True,pipe
3,nodo_2_ret_nodo_1_ret,7,6,None,4.700,1000.0,1000.0,0.1,0.0,0.742723,298.15,1,True,pipe
4,Tubo_Ida_nodo_2_cons_1_ida,2,3,None,4.600,1000.0,1000.0,0.1,0.0,0.742723,298.15,1,True,pipe
5,Tubo_bypass_1,3,8,None,0.002,1000.0,1000.0,0.1,0.0,0.742723,298.15,1,True,pipe
6,Tubo_Ida_cons_1_cons_2_ida,8,4,None,4.700,1000.0,1000.0,0.1,0.0,0.742723,298.15,1,True,pipe
7,Tubo_bypass_2,4,9,None,0.002,1000.0,1000.0,0.1,0.0,0.742723,298.15,1,True,pipe
8,Tubo_Retorno_nodo_2_cons_2_ret,9,7,None,4.500,1000.0,1000.0,0.1,0.0,0.742723,298.15,1,True,pipe


In [368]:
net.res_pipe

,v_mean_m_per_s,p_from_bar,p_to_bar,t_from_k,t_to_k,t_outlet_k,mdot_from_kg_per_s,mdot_to_kg_per_s,vdot_m3_per_s,reynolds,lambda,dp_friction_loss_bar
0,0.961355,3.500000,3.124817,310.000000,309.940173,309.940173,750.000000,-750.000000,0.755046,1.376771e+06,0.012020,0.375183
1,0.963239,1.843838,1.468055,315.148862,315.063045,315.063045,750.000000,-750.000000,0.756526,1.518190e+06,0.012016,0.375783
2,0.961337,3.124817,2.865503,309.940173,309.898999,309.898999,750.000000,-750.000000,0.755032,1.375429e+06,0.012020,0.259314
3,0.963268,2.103577,1.843838,315.208430,315.148862,315.148862,750.000000,-750.000000,0.756549,1.520193e+06,0.012016,0.259739
4,0.961323,2.865503,2.611709,309.898999,309.858841,309.858841,750.000000,-750.000000,0.755021,1.374351e+06,0.012020,0.253794
5,0.960535,2.611709,2.611599,309.858841,309.854766,309.858823,749.391355,-749.391355,0.754403,1.372703e+06,0.012020,0.000110
6,0.961307,2.611599,2.352292,309.854766,309.813890,309.813890,750.000000,-750.000000,0.755009,1.373170e+06,0.012020,0.259308
7,0.437235,2.352292,2.352269,309.813890,315.265659,309.813852,341.127985,-341.127985,0.343404,6.243227e+05,0.012076,0.000023
8,0.963290,2.352269,2.103577,315.265659,315.208430,315.208430,750.000000,-750.000000,0.756566,1.521806e+06,0.012016,0.248691


In [369]:
results_1=HP_1_object.get_main_results()
results_2=HP_2_object.get_main_results()

In [370]:
print(results_1)

{'Q_consumer': 15000.0, 'Q_network': 12720.331772587833, 'W_compressor': 2279.6682274122186, 'COP': 6.579904838620905}


In [371]:
15000.0/1685.6799401256844

8.898486386971895

In [372]:
print(results_2)

{'Q_consumer': 15000000.0, 'Q_network': 17092187.136565235, 'COP': 7.1695307450487755}


In [373]:
HP_1_object.get_results_solver()


##### RESULTS (CycleCloser) #####
+-------------+------------------+-------------------+
|             |   mass_deviation |   fluid_deviation |
|-------------+------------------+-------------------|
| CycleCloser |         0.00e+00 |          0.00e+00 |
+-------------+------------------+-------------------+
##### RESULTS (Compressor) #####
+-----------+----------+----------+-----------+----------+
|           |        P |       pr |        dp |    eta_s |
|-----------+----------+----------+-----------+----------|
| compresor | 2.28e+03 | 2.70e+00 | -1.19e+01 | 9.50e-01 |
+-----------+----------+----------+-----------+----------+
##### RESULTS (Condenser) #####
+-------------+-----------+----------+----------+----------+----------+----------+----------+-----------+----------+----------+----------+----------+------------+----------+------------+----------+-----------+------------+-----------+
|             |         Q |       UA |       kA |   td_log |     lmtd |    ttd_u |    ttd_l |  

In [374]:
HP_2_object.get_results_solver()


##### RESULTS (CycleCloser) #####
+-------------+------------------+-------------------+
|             |   mass_deviation |   fluid_deviation |
|-------------+------------------+-------------------|
| CycleCloser |         0.00e+00 |          0.00e+00 |
+-------------+------------------+-------------------+
##### RESULTS (Compressor) #####
+-----------+----------+----------+-----------+----------+
|           |        P |       pr |        dp |    eta_s |
|-----------+----------+----------+-----------+----------|
| compresor | 2.09e+06 | 2.40e+00 | -8.02e+00 | 9.50e-01 |
+-----------+----------+----------+-----------+----------+
##### RESULTS (Condenser) #####
+-------------+-----------+----------+----------+----------+----------+----------+----------+-----------+----------+----------+----------+----------+------------+----------+------------+----------+-----------+------------+-----------+
|             |         Q |       UA |       kA |   td_log |     lmtd |    ttd_u |    ttd_l |  